In [1]:
# Importing libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Display: do not truncate columns to clearly view the raw values
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [2]:
data =pd.read_excel(r"..\1.DATA\depenses_mars2025.xlsx")

---
# 1. Data Cleaning

---

In [3]:
data.sample(3)

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
250,2025-03-11,Achat imprimante,"466,41",Matériel,Darty,Espèces
363,2025-03-15,Courses événement,146.46,Restauration,Carrefour Pro,CB
66,2025-03-03,Ramettes papier,18.31,Fournitures,Papeterie Plus,Virement


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 741 entries, 0 to 740
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         741 non-null    object
 1   Libellé      741 non-null    object
 2   Montant      741 non-null    object
 3   Catégorie    737 non-null    object
 4   Fournisseur  741 non-null    object
 5   Paiement     741 non-null    object
dtypes: object(6)
memory usage: 34.9+ KB


---
### Reading `df.info()`

- **Total number of rows**: 741
- **Columns**: 6
- **Missing values detected**: 4 in the Category column
- **Data types**:
  - `Date`: object (will be converted to datetime)
  - `Montant`: object (will be converted to numeric)
  - `Catégorie`: object
  - `Fournisseur`: object
  - `Paiement`: object (will be converted to numeric)
---

In [5]:
data.describe(include='all')

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
count,741,741,741.00,737,741,741
unique,31,31,719.00,7,11,3
top,2025-03-31,Recharge carte SIM,45.81,Fournitures,Fnac,CB
freq,33,35,3.00,210,85,274


---
### Reading `df.describe(include='all')`

- **`Date`** — `unique=31` (consistent with the month of March); to be converted to date format; no NaNs
- **`Libellé`** — `unique=31` High repetition for the "Recharge carte SIM" service (35); no NaNs; to be left as object format.
- **`Montant`** — `unique = 719` (logical), to be converted to numeric (prior cleaning required); no NaNs
- **`Catégorie`** — 4 missing values (4 NaNs detected); Standardization to be checked to ensure there are 7 distinct values; to be converted to 'category' format
- **`Fournisseur`** — 11 suppliers for 741 rows. The top supplier is Fnac; no NaNs.
- **`Paiement`** — `unique = 3`, credit card (CB) is the preferred method; no NaNs; to be converted to 'category' format
---

In [6]:
# I create a copy of the dataframe (I do not touch the original):
data_clean=data.copy()

---
### Detection of abnormal values in future numeric columns currently formatted as object:
- Before any conversion with pd.to_numeric of future numeric columns still formatted as object, we identify abnormal values (special characters (?, commas instead of periods, euro symbols, etc.)).

---

In [7]:
for col in ['Montant']:
    serie = data_clean[col]
    # We only keep values of type str (the others are already numbers)
    non_num = serie[serie.apply(lambda x: isinstance(x, str))]
    print(f"--- {col} : {len(non_num)} valeur(s) en format texte ---")
    print(non_num.unique()[:10])
    print()

--- Montant : 121 valeur(s) en format texte ---
['95,28' '340,33' '75,67' '541,26' '79,60' '398,17' '43,89' '98,83'
 '25,76' '52,58']



---
### Diagnostic

We can see that the `Montant` column contains a **comma** as a decimal separator (French format);

We must therefore clean the 'commas' by replacing them with 'periods' to avoid getting NaNs that would distort our statistical results.

On the other hand, we do not observe any particular special characters (no ? or other characters).
Therefore, no special character cleaning is necessary.

---

---
### Cleaning character strings (object columns):
- Removal of unnecessary spaces in `Libellé`, `Catégorie`, `Fournisseur`, `Paiement` using str.strip()
- Standardize columns to "Title Case" format using title()
---

In [8]:
data_clean[['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']].head()  # Before treatment

,Libellé,Catégorie,Fournisseur,Paiement
0,Déplacement pro,Transport,Uber,CB
1,Transport aéroport,Transport,Uber,CB
2,Envoi colis,Logistique,Poste,Espèces
3,Billet train pro,Transport,SNCF,CB
4,Matériel informatique,Fournitures,Amazon,Virement


In [35]:
# Removing unnecessary spaces in text columns
cols_to_clean = ['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']
for col in cols_to_clean:
    data_clean[col] = data_clean[col].str.strip()

# Consistent capitalization of columns:
cols_to_title = ['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']
for col in cols_to_title:
    data_clean[col] = data_clean[col].str.title()

In [10]:
data_clean[['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']].head()  # After treatment

,Libellé,Catégorie,Fournisseur,Paiement
0,Déplacement Pro,Transport,Uber,Cb
1,Transport Aéroport,Transport,Uber,Cb
2,Envoi Colis,Logistique,Poste,Espèces
3,Billet Train Pro,Transport,Sncf,Cb
4,Matériel Informatique,Fournitures,Amazon,Virement


If we had detected special characters such as ? in futur numerical columns, we would have cleaned the column right here, before the standardization step of the categorical modalities.
We would have done, for example:
df['primes'] = df['primes'].replace('?', np.nan)

---
### Standardization of Categorical Modalities:

---

### We look at the unique modalities for each categorical column to spot similar modalities in order to standardize them:

In [11]:
# Unique modalities before standardization
print(data_clean['Libellé'].unique())
print()
print(data_clean['Catégorie'].unique())
print()
print(data_clean['Fournisseur'].unique())
print()
print(data_clean['Paiement'].unique())

['Déplacement Pro' 'Transport Aéroport' 'Envoi Colis' 'Billet Train Pro'
 'Matériel Informatique' 'Achat Ordinateur' 'Câble Secteur'
 'Trajet Client' 'Achat Repas Réunion' 'Carnets Et Blocs-Notes'
 'Courses Événement' 'Affranchissement' 'Déplacement Lyon'
 'Recharge Carte Sim' 'Achat Stylos' 'Lettre Suivie' 'Achat Câbles Hdmi'
 'Snacks Équipe' 'Abonnement Mobile' 'Achat Matériel Informatique'
 'Écran Professionnel' 'Claviers Et Souris' 'Ramettes Papier'
 'Disques Durs Externes' 'Achat Imprimante' 'Matériel Bureautique'
 'Matériel It' 'Train Client' 'Achat Tablette' 'Frais Téléphone'
 'Achat Fournitures De Bureau']

['Transport' 'Logistique' 'Fournitures' 'Informatique' 'Matériel'
 'Restauration' 'Télécom' nan]

['Uber' 'Poste' 'Sncf' 'Amazon' 'Boulanger' 'Darty' 'Carrefour Pro'
 'Papeterie Plus' 'Orange' 'Fnac' 'Am@Zon']

['Cb' 'Espèces' 'Virement']


--- 
### Diagnostic
- For the `Libellé` column, we can standardize "Matériel IT" and "Achat matériel informatique" into "Matériel informatique".
- For the `Fournisseur` column, we can standardize "Am@zon" into "Amazon".

---

In [12]:
data_clean['Libellé'] = data_clean['Libellé'].str.lower().str.strip()
data_clean['Libellé'] = data_clean['Libellé'].replace({
    'matériel it': 'Matériel Informatique',
    'achat matériel informatique': 'Matériel Informatique',
    'matériel informatique': 'Matériel Informatique',
})

data_clean['Fournisseur'] = data_clean['Fournisseur'].str.lower().str.strip()
data_clean['Fournisseur'] = data_clean['Fournisseur'].replace({
    'am@zon': 'Amazon',
    'amazon': 'Amazon',
})



In [13]:
# Unique modalities after standardization
print(data_clean['Libellé'].unique())
print()
print(data_clean['Catégorie'].unique())
print()
print(data_clean['Fournisseur'].unique())
print()
print(data_clean['Paiement'].unique())

['déplacement pro' 'transport aéroport' 'envoi colis' 'billet train pro'
 'Matériel Informatique' 'achat ordinateur' 'câble secteur'
 'trajet client' 'achat repas réunion' 'carnets et blocs-notes'
 'courses événement' 'affranchissement' 'déplacement lyon'
 'recharge carte sim' 'achat stylos' 'lettre suivie' 'achat câbles hdmi'
 'snacks équipe' 'abonnement mobile' 'écran professionnel'
 'claviers et souris' 'ramettes papier' 'disques durs externes'
 'achat imprimante' 'matériel bureautique' 'train client' 'achat tablette'
 'frais téléphone' 'achat fournitures de bureau']

['Transport' 'Logistique' 'Fournitures' 'Informatique' 'Matériel'
 'Restauration' 'Télécom' nan]

['uber' 'poste' 'sncf' 'Amazon' 'boulanger' 'darty' 'carrefour pro'
 'papeterie plus' 'orange' 'fnac']

['Cb' 'Espèces' 'Virement']


In [14]:
# We capitalize the columns after standardization to avoid having only lowercase:
cols_to_title = ['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']
for col in cols_to_title:
    data_clean[col] = data_clean[col].str.title()

In [15]:
data_clean[['Libellé', 'Catégorie', 'Fournisseur', 'Paiement']].head()  # We verify

,Libellé,Catégorie,Fournisseur,Paiement
0,Déplacement Pro,Transport,Uber,Cb
1,Transport Aéroport,Transport,Uber,Cb
2,Envoi Colis,Logistique,Poste,Espèces
3,Billet Train Pro,Transport,Sncf,Cb
4,Matériel Informatique,Fournitures,Amazon,Virement


---
### We correct the data types before proceeding with the conversion

- Here we must change commas to periods for the `Montant` column, then convert it to numeric
- We convert the Date to 'date' format
- We convert the object-type columns `Catégorie` and `Paiement` to 'category' type
---

In [16]:
# Conversion of the Date column to 'Date' format:
data_clean['Date']=pd.to_datetime(data_clean['Date'], errors = 'coerce')

# Cleaning and conversion of the 'Montant' column to numeric
# I could have also done:
# df['Montant'] = df['Montant'].astype(str)
# df['Montant'] = df['Montant'].str.strip()
# df['Montant'] = df['Montant'].str.replace(" ", "", regex=False)
# df['Montant'] = df['Montant'].str.replace(",", ".", regex=False)
# df['Montant'] = df['Montant'].str.replace("€", "", regex=False)
# df['Montant'] = df['Montant'].str.replace("?", "", regex=False)


for col in ['Montant']:
    data_clean[col] = (
        data_clean[col].astype(str)
                       .str.replace(",", ".", regex=False)
                       .str.replace("€", "", regex=False)
                       .str.replace(" ","",regex=False)
                       .str.strip()
    )
    data_clean[col] = pd.to_numeric(data_clean[col], errors='coerce')


# Conversion of 'Catégorie' and 'Paiement' columns to categorical variables
data_clean['Catégorie']  = data_clean['Catégorie'].astype('category')
data_clean['Paiement'] = data_clean['Paiement'].astype('category')

data_clean.dtypes


Date           datetime64[ns]
Libellé                object
Montant               float64
Catégorie            category
Fournisseur            object
Paiement             category
dtype: object

---
### Let's verify that we have successfully transformed commas into periods for the Montant column
---

In [17]:
data_clean['Montant']

0       86.97
1       59.20
2       20.84
3       88.38
4      104.46
        ...  
736     95.00
737    318.48
738    261.77
739    287.56
740    141.79
Name: Montant, Length: 741, dtype: float64

---
## Identify and handle missing values

---

In [18]:
# Identifying missing values (NaN or None)
print('before treatment :\n', data_clean.isna().sum())


before treatment :
 Date           0
Libellé        0
Montant        0
Catégorie      4
Fournisseur    0
Paiement       0
dtype: int64


---
The column with missing values is:
 
- 'Catégorie' with 4 missing values

We must therefore handle this column.


Imputation strategy:

| Column | Strategy | Justification |
|---|---|---|
| `Catégorie` | `'Not_specified'` (new category) | Categorical |

---

In [19]:
# For the numeric column: No numeric column to handle

# For the categorical column:
# Note: our 'Catégorie' column is in category format, so we need to include
# the cat.add_categories line
data_clean['Catégorie']=data_clean['Catégorie'].cat.add_categories('Not_specified')
data_clean['Catégorie']=data_clean['Catégorie'].fillna('Not_specified')

In [20]:
# Identifying missing values (NaN or None)
print('After treatment :\n', data_clean.isna().sum())

After treatment :
 Date           0
Libellé        0
Montant        0
Catégorie      0
Fournisseur    0
Paiement       0
dtype: int64


---
### Duplicate detection and elimination:

---

In [21]:
# Number of Duplicates detection
print("\nNumber of duplicates:",data_clean.duplicated().sum())


Number of duplicates: 13


In [22]:
# Overview of duplicates:
duplicates=data_clean[data_clean.duplicated]
duplicates

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
728,2025-03-31,Câble Secteur,354.16,Matériel,Darty,Cb
729,2025-03-31,Billet Train Pro,44.07,Transport,Sncf,Cb
730,2025-03-31,Frais Téléphone,71.64,Télécom,Orange,Cb
731,2025-03-31,Achat Câbles Hdmi,56.53,Fournitures,Amazon,Virement
732,2025-03-31,Achat Ordinateur,443.63,Informatique,Boulanger,Virement
733,2025-03-31,Transport Aéroport,69.24,Transport,Uber,Cb
734,2025-03-31,Affranchissement,31.99,Logistique,Poste,Virement
735,2025-03-31,Abonnement Mobile,39.21,Télécom,Orange,Cb
736,2025-03-31,Abonnement Mobile,95.00,Télécom,Orange,Virement
737,2025-03-31,Achat Câbles Hdmi,318.48,Fournitures,Amazon,Virement


In [23]:
# Elimination of duplicates, we will base it on 4 columns ("Date", "Libellé", "Montant", "Fournisseur")
data_clean = data_clean.drop_duplicates(subset=['Date','Libellé','Montant','Fournisseur'])
print('rows remaining :', len(data_clean))

rows remaining : 728


In [24]:
# We verify that we indeed deleted the duplicates :
print("Number of duplicates:",data_clean.duplicated().sum())

Number of duplicates: 0


---
### Anomaly detection
We will first identify (without deleting them) the anomalies, notably:
- Negative amounts, which may indicate refunds or data entry errors.
- Abnormally high amounts (greater than €3,000), which must be flagged and analyzed.

---

In [25]:
# Identification of negative amounts and amounts over 3,000 using filtering
anomalies = data_clean[(data_clean['Montant'] < 0) | (data_clean['Montant'] > 3000)]
display(anomalies)

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
7,2025-03-01,Câble Secteur,9465.78,Matériel,Darty,Cb
19,2025-03-02,Recharge Carte Sim,-35.35,Télécom,Orange,Virement
166,2025-03-07,Achat Ordinateur,10036.47,Informatique,Boulanger,Virement
252,2025-03-11,Abonnement Mobile,-49.20,Télécom,Orange,Espèces
274,2025-03-12,Transport Aéroport,14816.20,Transport,Uber,Espèces
308,2025-03-13,Ramettes Papier,-132.91,Fournitures,Papeterie Plus,Virement
337,2025-03-14,Transport Aéroport,8307.89,Transport,Uber,Virement
491,2025-03-21,Courses Événement,13584.31,Restauration,Carrefour Pro,Virement
502,2025-03-22,Lettre Suivie,-32.00,Logistique,Poste,Virement
538,2025-03-23,Déplacement Pro,13304.52,Transport,Uber,Cb


---
# 2. Data Analysis

---

---
### A. Data_Cleaning :

---

---
### Duplicates

---

During the cleaning step, I identified and removed 13 duplicates to avoid overestimating total expenditures.

---
### Categories completed manually (Missing values)

---

In [26]:
Cat_missing = data_clean[data_clean['Catégorie'] == 'Not_specified']
display(Cat_missing)

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
295,2025-03-12,Billet Train Pro,34.86,Not_specified,Sncf,Cb
325,2025-03-14,Train Client,44.76,Not_specified,Sncf,Espèces
376,2025-03-16,Abonnement Mobile,95.18,Not_specified,Orange,Cb
406,2025-03-17,Frais Téléphone,132.36,Not_specified,Orange,Cb


These 4 rows had missing values. They were labeled 'Non_renseigne' to maintain data integrity.

---
### B. Descriptive Statistics

---

---
### Global statistics on amounts

---

In [27]:
data_clean.describe(include='number').round(2)

,Montant
count,728.00
mean,285.16
std,1261.05
min,-587.63
25%,45.87
50%,96.12
75%,222.30
max,14982.97


### Statistical Interpretation:

The statistical analysis of the 'Montant' (Amount) column reveals major disparities:

- A strongly right-skewed distribution, with a median of €96.12 that is significantly lower than the mean (€285.16). This indicates that a large majority (50%) of the expenses involve small amounts (under €96.12). The mean is potentially pulled upward by extreme values, as suggested by the maximum value of €14,982.97. Therefore, the mean does not reflect the daily reality.

- A massive dispersion, reflected by a standard deviation of €1,261.05. This confirms that the expenses are highly heterogeneous, with disproportionate gaps between anomalies and routine expenses.


### Business Interpretation:

These initial descriptive results suggest that the expense file contains major anomalies that distort the overall budget analysis. 
While the average cost per expense appears high (€285.16), this inflation seems driven by suspicious transactions or data entry errors. The median demonstrates that the company's actual typical purchasing behavior is closer to €96 per transaction.

This supports the hypothesis that data entry errors, refunds, or suspicious transactions are likely present in the dataset. However, this hypothesis must be confirmed by an isolated analysis of negative amounts and amounts exceeding €3,000 before drawing any conclusions regarding the actual budget for the month.

---
### Analysis of amounts greater than 3,000 euros:

---

In [28]:
expenses_greater = data_clean[data_clean['Montant'] > 3000]
display(expenses_greater)

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
7,2025-03-01,Câble Secteur,9465.78,Matériel,Darty,Cb
166,2025-03-07,Achat Ordinateur,10036.47,Informatique,Boulanger,Virement
274,2025-03-12,Transport Aéroport,14816.20,Transport,Uber,Espèces
337,2025-03-14,Transport Aéroport,8307.89,Transport,Uber,Virement
491,2025-03-21,Courses Événement,13584.31,Restauration,Carrefour Pro,Virement
538,2025-03-23,Déplacement Pro,13304.52,Transport,Uber,Cb
680,2025-03-29,Billet Train Pro,14982.97,Transport,Sncf,Cb
689,2025-03-30,Lettre Suivie,10702.67,Logistique,Poste,Cb


In [29]:
expenses_greater.describe(include='number').round(2)

,Montant
count,8.00
mean,11900.10
std,2578.86
min,8307.89
25%,9893.80
50%,12003.60
75%,13892.28
max,14982.97


### Statistical Interpretation:

The average amount of the observed anomalies is €11,900.10.
The standard deviation of €2,578.86 shows a high dispersion, even among these high values.
The median of these amounts is around €12,000, indicating that 50% of these large amounts are greater than €12,003.60, which confirms that these transactions are due to repeated data entry errors.

### Business Interpretation:

The isolated analysis of these amounts confirms the hypothesis that they are due to data entry errors, as it is technically impossible to pay such high amounts for Ubers, registered letters, or train tickets.

---
### Most expensive expense categories 
- Comparison across the entire dataset and on anomalies (expenses > €3,000)
---

### Most expensive categories across the entire dataset
---

In [30]:
data_clean.groupby('Catégorie')['Montant'].sum().round(2).sort_values(ascending=False)

Catégorie
Transport        60711.40
Fournitures      43462.92
Matériel         32690.07
Informatique     31787.43
Restauration     19309.67
Logistique       12212.25
Télécom           7116.13
Not_specified      307.16
Name: Montant, dtype: float64

---
### Most expensive categories for expenses greater than 3,000 euros
---

In [31]:
expenses_greater=data_clean[data_clean['Montant'] > 3000]

In [32]:
expenses_greater.groupby('Catégorie')['Montant'].sum().round(2).sort_values(ascending=False)

Catégorie
Transport        51411.58
Restauration     13584.31
Logistique       10702.67
Informatique     10036.47
Matériel          9465.78
Fournitures          0.00
Télécom              0.00
Not_specified        0.00
Name: Montant, dtype: float64

---
### Statistical Interpretation:
Across the entire dataset, the "Transport" category comes out on top with a total expenditure of €60,711.40.
However, comparing this with the isolated analysis of amounts greater than €3,000 reveals that this same category accounts for €51,411.58 of the anomalies. This means that these outlier transactions represent approximately 85% of the transport budget.

### Business Interpretation:
Transport appears to be the primary expense item, but this is not actually the case as it is driven by data entry errors.
If we exclude these anomalies, the true primary expense item is "Fournitures" (Supplies).
It is therefore imperative to manually correct these data entry errors.

---

---
### Analysis of expenses with negative amounts

---

In [33]:
negative_expenses=data_clean[data_clean['Montant'] < 0]
display(negative_expenses)

,Date,Libellé,Montant,Catégorie,Fournisseur,Paiement
19,2025-03-02,Recharge Carte Sim,-35.35,Télécom,Orange,Virement
252,2025-03-11,Abonnement Mobile,-49.20,Télécom,Orange,Espèces
308,2025-03-13,Ramettes Papier,-132.91,Fournitures,Papeterie Plus,Virement
502,2025-03-22,Lettre Suivie,-32.00,Logistique,Poste,Virement
582,2025-03-25,Matériel Bureautique,-587.63,Matériel,Darty,Virement


---
We identify 5 transactions with a negative amount, for a cumulative total of -€836.09.

These rows likely correspond to supplier refunds or credit notes.
I recommend verifying that these amounts are not sign errors.

---

In [34]:
data_clean.to_excel("expenses_march2025_Clean.xlsx", index=False)